# Edisi 003: Provinsi makin kaya, warganya makin sejahtera?

## Pertanyaan

Kalimat itu sering terlontar apa adanya: provinsi yang kaya, pasti warganya sejahtera. Rakyat makmur. Sebelum percaya, saya bawa angkanya: PDRB per kapita dan Indeks
Pembangunan Manusia (IPM) untuk 34 provinsi di tahun 2021. Seberapa erat kaitan keduanya, dan
seberapa jauh sebenarnya selisih antarprovinsi itu?

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

from tools import gaya

gaya.terapkan()
df = pd.read_csv("data/provinsi.csv")
d = df.dropna(subset=["pdrb_kapita_juta_2021", "ipm_2021"]).copy()
x, y = d.pdrb_kapita_juta_2021, d.ipm_2021
print("provinsi lengkap:", len(d))
df.head(3)

provinsi lengkap: 34


,provinsi,kunci,pdrb_kapita_juta_2021,pdrb_kapita_usd_2021,wilayah_geografis,jawa,ipm_2021,ipm_2024,ipm_2025
0,Daerah Khusus Jakarta,jakarta,274.71,19199.0,Jawa,True,81.11,84.15,85.05
1,Yogyakarta,yogyakarta,40.23,2812.0,Jawa,True,80.22,81.62,82.48
2,Kalimantan Timur,kalimantan timur,182.54,12757.0,Kalimantan,False,76.88,78.79,79.39


## Data & cara ukur

Yang diukur, dua angka untuk tiap provinsi di 2021. **PDRB per kapita**: nilai seluruh barang
dan jasa yang dihasilkan satu provinsi dalam setahun, dibagi jumlah penduduknya, dalam juta
rupiah. **IPM**: indeks 0-100 gabungan umur panjang, lama sekolah, dan daya beli, ukuran
resmi "sejahtera" versi BPS. Pasangannya sengaja satu tahun yang sama supaya adil.

Uji kelayakan sumber: situs BPS memblokir akses otomatis (403), jadi yang lulus adalah tabel
Wikipedia bahasa Indonesia yang merujuk BPS; mentahnya disimpan apa adanya untuk
diaudit. Yang hilang sejak awal: ini data agregat provinsi. PDRB per
kapita bukan gaji, dan 2021 adalah tahun terakhir yang punya PDRB per kapita di sumber ini,
sebelum empat provinsi Papua dimekarkan (PDRB 34 provinsi, IPM terbaru sudah 38).

In [2]:
ringkas = pd.DataFrame({"pdrb_kapita_juta_2021": x, "ipm_2021": y}).describe().T[
    ["count", "mean", "std", "min", "max"]].round(2)
print("terkaya:", d.loc[x.idxmax(), "provinsi"], f"{x.max():.1f} juta | termiskin:",
      d.loc[x.idxmin(), "provinsi"], f"{x.min():.1f} juta")
print("IPM tertinggi:", d.loc[y.idxmax(), "provinsi"], f"{y.max():.2f} | terendah:",
      d.loc[y.idxmin(), "provinsi"], f"{y.min():.2f}")
ringkas

terkaya: Daerah Khusus Jakarta 274.7 juta | termiskin: Nusa Tenggara Timur 20.6 juta
IPM tertinggi: Daerah Khusus Jakarta 81.11 | terendah: Papua 60.62


,count,mean,std,min,max
pdrb_kapita_juta_2021,34.0,66.86,51.61,20.58,274.71
ipm_2021,34.0,71.43,3.87,60.62,81.11


## Pembedahan 1: Seberapa erat kaitannya?

Alat pikir edisi ini: **korelasi** dan **R²**. Korelasi Pearson (r) mengukur seberapa erat dua
hal berjalan bersama, dari -1 (berlawanan sempurna) sampai 1 (sejalan sempurna). R²
menerjemahkannya: berapa banyak keragaman satu hal yang bisa dijelaskan yang lain, dalam skala
0 sampai 100 persen.

Di sini r = **0,48** (CI95% 0,17 s.d. 0,71; p = 0,002) dengan R² = **0,23**. Artinya kaitannya
positif tapi lemah: penghasilan daerah hanya menjelaskan **23%** keragaman kesejahteraan;
77% sisanya dijelaskan hal lain. (Sumbu penghasilan memakai skala kali-lipat, setiap langkah
sama berarti kali lipat, bukan tambah tetap, karena Jakarta begitu jauh di depan.)

Titik-titik yang menjauh dari garis justru ceritanya: **Yogyakarta** dengan PDRB per kapita
cuma 40,2 juta (di bawah rata-rata) justru IPM-nya 80,2, peringkat dua nasional, 9,8 poin di
atas dugaan garis. Di ujung lain, **Papua** 10,3 poin di bawah dugaan.

In [3]:
lx = np.log10(x)
lreg = stats.linregress(lx, y)
r = lreg.rvalue
n = len(d)
z_ = np.arctanh(r)
se_z = 1 / np.sqrt(n - 3)
lo_r, hi_r = np.tanh(z_ - 1.96 * se_z), np.tanh(z_ + 1.96 * se_z)

d["dugaan"] = lreg.intercept + lreg.slope * lx
d["sisa"] = y - d.dugaan

pd.DataFrame({"nilai": [
    f"r = {r:.3f} (CI95 {lo_r:.3f} s.d. {hi_r:.3f}) p = {lreg.pvalue:.4f}  [skala log10]",
    f"R2 = {r ** 2:.3f}  ({100 * r ** 2:.0f}% keragaman IPM terjelaskan)",
    f"Yogyakarta: +{d.loc[d.provinsi == 'Yogyakarta', 'sisa'].iloc[0]:.1f} poin di atas garis",
    f"Papua: {d.loc[d.provinsi == 'Papua', 'sisa'].iloc[0]:.1f} poin di bawah garis",
]}, index=["korelasi Pearson (transformasi Fisher)", "R kuadrat", "sisa terbesar", "sisa terkecil"])

,nilai
korelasi Pearson (transformasi Fisher),r = 0.484 (CI95 0.174 s.d. 0.706) p = 0.0038 ...
R kuadrat,R2 = 0.234 (23% keragaman IPM terjelaskan)
sisa terbesar,Yogyakarta: +9.9 poin di atas garis
sisa terkecil,Papua: -10.7 poin di bawah garis


In [4]:
def grafik_1(mode):
    fig, ax, fs = gaya.dasar(
        mode,
        "Seberapa erat kaitan kekayaan dengan kesejahteraan provinsi?",
        "R² = 0,23. Kaitan positif tapi lemah (r = 0,48; CI95 0,17 s.d. 0,71).",
        "Wikipedia (rujukan BPS), PDRB per kapita dan IPM 34 provinsi, 2021", 34,
    )
    ax.scatter(x, y, s=34, color=gaya.INK2, zorder=3)
    xs = np.logspace(np.log10(x.min() * 0.92), np.log10(x.max() * 1.08), 60)
    ax.plot(xs, lreg.intercept + lreg.slope * np.log10(xs), color=gaya.AKSEN, lw=2.3)
    for nama, geser in [("Yogyakarta", (8, 6)), ("Papua", (8, -14)), ("Daerah Khusus Jakarta", (-12, 8))]:
        row = d.loc[d.provinsi == nama].iloc[0]
        ax.annotate(row.provinsi.replace("Daerah Khusus ", ""),
                    (row.pdrb_kapita_juta_2021, row.ipm_2021),
                    textcoords="offset points", xytext=geser, ha="left",
                    fontsize=9 * fs, color=gaya.INK, fontweight="bold")
    ax.set_xscale("log")
    ax.set_xticks([25, 50, 100, 200])
    ax.set_xticklabels(["25", "50", "100", "200"])
    ax.set_xlabel("PDRB per kapita 2021 (juta Rp, skala kali-lipat)", fontsize=9.5 * fs)
    ax.set_ylabel("IPM 2021 (skala 0-100)", fontsize=9.5 * fs)
    ax.grid(True, axis="y")
    ax.grid(False, axis="x")
    gaya.simpan(fig, "01-kaitan-kaya-sejahtera", mode)


for mode in gaya.MODE:
    grafik_1(mode)

## Pembedahan 2: Berapa kali lipat selisihnya?

Korelasi bicara kekaitan; bagian ini bicara jarak. Ujung ke ujung, PDRB per kapita **13,3 kali
lipat**: Jakarta 274,7 juta per orang per tahun melawan Nusa Tenggara Timur 20,6 juta. IPM?
Hanya **1,34 kali lipat**: Jakarta 81,11 melawan Papua 60,62.

Perhatikan beda kedua lipatan itu. Keuangan daerah berjauhan sekali; hasil pembangunan yang
diukur umur panjang, sekolah, dan daya beli ternyata jauh lebih merata. Kaya memang enak,
tapi angkanya bilang: kaya tidak otomatis menjamin sejahtera, dan tidak kaya tidak otomatis
berarti tertinggal jauh.

In [5]:
imax, imin = x.idxmax(), x.idxmin()
jmax, jmin = y.idxmax(), y.idxmin()
lip_pdrb = x[imax] / x[imin]
lip_ipm = y[jmax] / y[jmin]

def grafik_2(mode):
    fig, ax, fs = gaya.dasar(
        mode,
        "Berapa kali lipat selisih antarprovinsi?",
        "Penghasilan 13,3 kali lipat (Jakarta vs NTT); IPM hanya 1,34 kali (Jakarta vs Papua).",
        "Wikipedia (rujukan BPS), 34 provinsi, 2021; tertinggi dibagi terendah", 34,
    )
    ax.bar(["PDRB per kapita", "IPM"], [lip_pdrb, lip_ipm],
           color=[gaya.AKSEN, gaya.INK2], width=0.5)
    ax.axhline(1, color=gaya.INK, lw=1.1, ls="--", label="tingkat setara (1x)")
    ax.text(0, lip_pdrb + 0.55, gaya.idn(lip_pdrb, 1) + "x", ha="center",
            fontsize=11 * fs, color=gaya.AKSEN, fontweight="bold")
    ax.text(0, lip_pdrb - 1.35, f"{d.loc[imax, 'provinsi']} vs\n{d.loc[imin, 'provinsi']}",
            ha="center", fontsize=8.5 * fs, color=gaya.KERTAS)
    ax.text(1, lip_ipm + 0.35, gaya.idn(lip_ipm, 2) + "x", ha="center",
            fontsize=11 * fs, color=gaya.INK2, fontweight="bold")
    ax.set_ylabel("Lipatan tertinggi terhadap terendah (kali)", fontsize=9.5 * fs)
    leg = ax.legend(loc="upper right", frameon=False, fontsize=9 * fs)
    for t in leg.get_texts():
        t.set_color(gaya.INK)
    ax.grid(True, axis="y")
    ax.grid(False, axis="x")
    gaya.simpan(fig, "02-lipatan-antarprovinsi", mode)


for mode in gaya.MODE:
    grafik_2(mode)

## Uji kekokohan: Apakah kesimpulannya dapat di pertahankan?

Karena BPS tidak punya API terbuka, silang sumber tidak mungkin; yang diuji adalah kekokohan
terhadap pilihan cara hitung. Pertama, korelasi dihitung ulang di skala lurus (tambah juta,
bukan kali lipat): r = 0,52 (CI95 0,22 s.d. 0,73). Kedua, uji Spearman yang hanya memakai
peringkat, tanpa asumsi bentuk hubungan: rho = 0,38 (p = 0,03). Ketiganya berbeda angka tapi
sepakat soal kesimpulan: kaitan kekayaan dan kesejahteraan itu nyata namun lemah, dan selalu
banyak pengecualian.

In [6]:
lurus = stats.linregress(x, y)
z1 = np.arctanh(lurus.rvalue)
lo1, hi1 = np.tanh(z1 - 1.96 * se_z), np.tanh(z1 + 1.96 * se_z)
sp = stats.spearmanr(x, y)
print(f"skala lurus : r = {lurus.rvalue:.3f} (CI95 {lo1:.3f} s.d. {hi1:.3f}) R2 = {lurus.rvalue ** 2:.3f}")
print(f"Spearman    : rho = {sp.statistic:.3f} p = {sp.pvalue:.4f}")

skala lurus : r = 0.522 (CI95 0.223 s.d. 0.731) R2 = 0.272
Spearman    : rho = 0.379 p = 0.0270


In [7]:
def kartu_teks():
    gaya.simpan(gaya.kartu("linkedin", "EDISI 003 · EKONOMI · OKTOBER 2026",
        "Provinsi makin kaya,\nwarganya makin sejahtera?",
        [(gaya.INK2, "Saya pasangkan PDRB per kapita\ndan IPM untuk 34 provinsi,\nsatu tahun yang sama: 2021."),
         (gaya.AKSEN, "(jawabannya di dalam)")],
        "Jurnal Eksplorasi Data Keseharian"), "slide-01-pertanyaan", "linkedin", penuh=True)

    gaya.simpan(gaya.kartu("linkedin", "EDISI 003 · EKONOMI",
        "Datanya dari mana?",
        [(gaya.INK, "Tabel Wikipedia rujukan BPS:\nPDRB per kapita 2021 dan IPM\n2021 (plus IPM 2024-2025)."),
         (gaya.INK2, "PDRB per kapita = nilai output\ndaerah per orang, bukan gaji.\nIPM = umur panjang, sekolah,\ndan daya beli (skala 0-100).\nBPS langsung memblokir robot\n(403), jadi lewat Wikipedia.")],
        "Jurnal Eksplorasi Data Keseharian"), "slide-02-data", "linkedin", penuh=True)

    gaya.simpan(gaya.kartu("linkedin", "EDISI 003 · EKONOMI",
        "Temuan, batas, dan sumber",
        [(gaya.AKSEN, "Kaitannya lemah: R² = 0,23.\nSelisih penghasilan 13,3 kali\nlipat, IPM hanya 1,34 kali."),
         (gaya.INK2, "Batas: agregat provinsi; tahun 2021;\nkorelasi bukan bukti sebab."),
         (gaya.INK2, "Sumber: Wikipedia (BPS), CC BY,\ndiakses 22 Sep 2026.")],
        "Jurnal Eksplorasi Data Keseharian"), "slide-05-batas", "linkedin", penuh=True)


kartu_teks()
print("karousel:", sorted(p.name for p in Path("linkedin/gambar").glob("*.png")))

karousel: ['01-kaitan-kaya-sejahtera.png', '02-lipatan-antarprovinsi.png', 'slide-01-pertanyaan.png', 'slide-02-data.png', 'slide-05-batas.png']


## Temuan

> **Kaitan ada tapi lemah: kekayaan daerah hanya menjelaskan 23% keragaman kesejahteraan, dan
> selisih penghasilan antarprovinsi 13,3 kali lipat sementara selisih IPM hanya 1,34 kali lipat.**

Kalimat "provinsi kaya pasti sejahtera" tidak salah sepenuhnya, hanya jauh lebih lemah dari
yang dibayangkan. Sisanya kelihatan dari para pengecualian: Yogyakarta dengan PDRB per kapita
40,2 juta menempati peringkat dua IPM nasional; Papua, walau penghasilannya setara banyak
provinsi lain, IPM-nya 10,3 poin di bawah dugaan.

## Batas & cara reproduksi

Yang tidak boleh disimpulkan: penyebabnya, dan kantong siapa yang tebal. PDRB per kapita adalah
output daerah per orang, bukan pendapatan warga; provinsi migas contohnya bisa sangat kaya
angkanya tanpa otomatis IPM tinggi. IPM sendiri hanya tiga dimensi. Datanya agregat 34 provinsi
tahun 2021, sebelum pemekaran Papua; tabel Wikipedia bisa berubah sewaktu-waktu dan datanya
kita simpan apa adanya untuk diaudit. Hubungan sebab-akibat butuh penelitian sendiri.

```bash
python3 data_scraping.py
python3 -m nbconvert --to notebook --execute --inplace analisis.ipynb
```

## Sumber Data

- Wikipedia bahasa Indonesia (CC BY-SA): "Daftar provinsi di Indonesia menurut PDRB", "Daftar provinsi Indonesia menurut IPM", "Provinsi di Indonesia" (rujukan BPS), diakses 22 September 2026.
- Data olahan: `data/provinsi.csv`. Kode pengambilan: `tools/data_scraping.py`.
- Data mentah apa adanya: `data/mentah/`.